# Lab 03 Extra: Multi-Agent LLM Mafia

This is a companion demo to `Lab_03_Knowledge.ipynb`, not a replacement for it.

In the main Lab 03 notebook, we represented knowledge with **propositional logic** and used a **model checker** (`sympy.logic`) to prove facts in the Cluedo game — every rule was hand-encoded, and the machinery guaranteed a correct answer.

Here, six separate LLM instances play a full game of **Mafia** against each other. Nobody hand-codes the inference rules this time. Each AI agent has to track *who knows what*, notice contradictions, and reason about hidden state using nothing but natural language — the same underlying problem as Cluedo, but solved (and exploited) very differently.

Watch for:
- **Knowledge inference** — agents inferring hidden roles from public statements and voting patterns.
- **Deception** — the Mafia agent is explicitly told it may lie. Watch its private reasoning vs. its public statement.
- **Multi-agent collaboration (and its failure modes)** — Village-aligned agents must coordinate with no shared ground truth, and sometimes get it badly wrong.
- **Model intelligence, side by side** — this game's roster deliberately mixes 3 flagship-tier models with 3 budget-tier models. Watch for differences in how convincingly each argues, lies, or gets caught.

## Why this is a replay, not a live demo

This notebook **does not call any LLM API**. Every game shown here was generated ahead of time by `generate_games.py` and saved as a JSON file in `games/`. This notebook only reads and displays that file.

Why: live API calls in front of a class risk rate limits, timeouts, unpredictable pacing, and real cost every time the notebook is re-run. Pre-generating games lets us pick good ones ahead of time and replay them instantly and for free. See `DESIGN.md` in this folder for the full reasoning.

**Note:** the game picker below uses an interactive dropdown (`ipywidgets`), so it needs to be run live in Jupyter or Colab — it will show as an empty placeholder in a statically-exported HTML/PDF version of this notebook, or in a screenshot taken before you've selected a game. That's expected; just run the cells live.

If you have your own OpenRouter API key, you can generate new games yourself — see the last section of this notebook.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import replay_utils as ru

## The roster

Every game uses the same fixed 6-model roster: 3 flagship-tier models and 3 budget-tier models, spanning 4 vendors. Which *role* (Mafia / Detective / Doctor / Villager) each model plays is randomized per game — so across multiple games you can compare how the *same* model behaves under different roles, and how models of different tiers stack up against each other.

| Tier | Vendor | Model |
|---|---|---|
| Flagship | OpenAI | `openai/gpt-6-astra` |
| Flagship | Anthropic | `anthropic/claude-sonnet-5` |
| Flagship | Anthropic | `anthropic/claude-opus-5` |
| Budget | Google | `google/gemini-3.8-flash` |
| Budget | DeepSeek | `deepseek/deepseek-v4-flash-0731` |
| Budget | OpenAI | `openai/gpt-5.4-mini` |

**Rules (deliberately simple):** 6 players — 1 Mafia, 1 Detective, 1 Doctor, 3 Villagers. Each round: night actions (Doctor protects, Mafia kills, Detective investigates) → morning death announcement → one public statement per living player → a vote to eliminate someone. Mafia wins once Mafia count ≥ remaining Villagers; Village wins once all Mafia are eliminated.

**Roles are hidden throughout the game** and only revealed together at the very end — so a player who gets voted out mid-game isn't necessarily confirmed as the Mafia. Try guessing as you read!

## Play back a game

Pick any game from the dropdown below — each entry is one complete, independently-generated game, labeled with its round count, winner, and generation cost (none of that is a role/identity spoiler). Selecting one renders it immediately.

In [ ]:
browser = ru.GameBrowser("games", mode="full")
browser.show()

## Optional: step through a game round by round

For the 1–2 games you want to linger on and narrate in detail, pick a game from the dropdown below to reveal one round at a time behind a **Next round ▶** button. This is purely a pacing control — it costs nothing (it's just re-rendering already-generated text), so feel free to pause here and ask the class "who do you suspect right now?" before clicking through. There's no built-in prompt text for this; it's left to you as the instructor.

Pick a game, then click the button to advance.

In [ ]:
step_browser = ru.GameBrowser("games", mode="step")
step_browser.show()

---
## (Optional, for students with their own OpenRouter key) Generate a new game

If you want to generate a fresh game yourself instead of only replaying pre-made ones:

1. Get an OpenRouter API key from [openrouter.ai](https://openrouter.ai) and put it in a `.env` file as `OPENROUTER_API_KEY=...`.
2. From this folder, run:
   ```bash
   uv run python test_models.py      # confirms all 6 models are reachable first
   uv run python generate_games.py --n 1
   ```
3. A new file will appear in `games/`. Re-run the dropdown cells above (or scroll up and re-select) and your new game will show up as an option automatically.

A single game costs roughly $0.15–$0.40 depending on how many rounds it runs — each agent's prompt includes the full game history so far (that's what gives agents real cross-round memory), so cost grows with round count. Cost is dominated by the 3 flagship-tier agents; the 3 budget-tier agents are nearly free by comparison.

**Discussion prompt (optional):** after generating your own game, open the JSON file directly and compare each agent's `private_reasoning` to their `public_statement` in the day-discussion entries. Where did an agent say something different from what it was actually thinking? Was it lying, bluffing, or just being diplomatic?